In [ ]:
!pip install mne -q
import mne
print(f"MNE version : {mne.__version__}")

In [ ]:
# Téléchargement d'un sujet ERP CORE (paradigme P300 oddball)
from mne.datasets import erp_core

data_path = erp_core.data_path()
print(f"Données téléchargées dans : {data_path}")

In [ ]:
import os

# Explorer ce qui a vraiment été téléchargé
data_path = erp_core.data_path()
for root, dirs, files in os.walk(data_path):
    level = root.replace(str(data_path), '').count(os.sep)
    if level < 3:  # Juste les 3 premiers niveaux
        indent = '  ' * level
        print(f'{indent}{os.path.basename(root)}/')
        if level == 2:
            for f in files[:3]:  # Premiers fichiers seulement
                print(f'{indent}  {f}')

In [ ]:
import os

data_path = erp_core.data_path()
print("Path:", data_path)
print("Contenu:")
for item in os.listdir(data_path):
    print(" ", item)

In [ ]:
raw_file = os.path.join(data_path, 'ERP-CORE_Subject-001_Task-Flankers_eeg.fif')

raw = mne.io.read_raw_fif(raw_file, preload=True)
print(raw.info)

In [ ]:
from IPython.display import Image

raw.plot(duration=10, n_channels=33, show=False).savefig('raw_signal.png', dpi=150, bbox_inches='tight')
Image('raw_signal.png')

In [ ]:
# Filtre passe-bande : on garde entre 0.1 Hz et 40 Hz
raw_filtered = raw.copy().filter(l_freq=0.1, h_freq=40.0)
print("Filtrage terminé")

In [ ]:
# Lire les événements (marqueurs = timestamps des stimuli)
events, event_id = mne.events_from_annotations(raw_filtered)
print("Types d'événements :", event_id)
print(f"Nombre total d'événements : {len(events)}")

In [ ]:
epochs = mne.Epochs(
    raw_filtered,
    events,
    event_id={
        'compatible/left':    3,
        'compatible/right':   4,
        'incompatible/left':  5,
        'incompatible/right': 6
    },
    tmin=-0.2,
    tmax=0.8,
    baseline=(-0.2, 0),
    reject_by_annotation=True,
    preload=True
)
print(epochs)

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import Image

erp_compatible   = epochs['compatible/left', 'compatible/right'].average()
erp_incompatible = epochs['incompatible/left', 'incompatible/right'].average()

fig, ax = plt.subplots(figsize=(10, 4))

ax.plot(erp_compatible.times * 1000,
        erp_compatible.get_data(picks='FCz')[0] * 1e6,
        label='Compatible', color='steelblue')

ax.plot(erp_incompatible.times * 1000,
        erp_incompatible.get_data(picks='FCz')[0] * 1e6,
        label='Incompatible', color='tomato')

ax.axvline(0, color='black', linestyle='--', alpha=0.5, label='Stimulus')
ax.axhline(0, color='black', linestyle='-', alpha=0.2)
ax.set_xlabel('Temps (ms)')
ax.set_ylabel('Amplitude (µV)')
ax.set_title('ERP — FCz — Compatible vs Incompatible')
ax.legend()
ax.invert_yaxis()
plt.tight_layout()

fig.savefig('erp.png', dpi=150, bbox_inches='tight')
Image('erp.png')

In [ ]:
import numpy as np

# Topomap à 3 moments clés : 100ms, 200ms, 300ms
times_to_plot = [0.1, 0.2, 0.3]

fig, axes = plt.subplots(2, 3, figsize=(12, 6))

for i, t in enumerate(times_to_plot):
    erp_compatible.plot_topomap(
        times=t,
        axes=axes[0, i],
        show=False,
        colorbar=False,
        vlim=(-8, 8)
    )
    axes[0, i].set_title(f'Compatible — {int(t*1000)}ms')

    erp_incompatible.plot_topomap(
        times=t,
        axes=axes[1, i],
        show=False,
        colorbar=False,
        vlim=(-8, 8)
    )
    axes[1, i].set_title(f'Incompatible — {int(t*1000)}ms')

plt.suptitle('Distribution spatiale de l\'activité EEG', fontsize=13)
plt.tight_layout()
fig.savefig('topomap.png', dpi=150, bbox_inches='tight')
Image('topomap.png')

In [ ]:
from scipy import stats
import numpy as np

# Fenêtre d'intérêt : 150-300ms (pic du N2)
tmin_stat, tmax_stat = 0.15, 0.30

# Extraire les données brutes par trial sur FCz
compat_data   = epochs['compatible/left', 'compatible/right'].copy()
incompat_data = epochs['incompatible/left', 'incompatible/right'].copy()

# Moyenne d'amplitude sur la fenêtre pour chaque trial
def mean_amplitude(ep, tmin, tmax, ch='FCz'):
    ep_crop = ep.copy().crop(tmin=tmin, tmax=tmax)
    idx = ep_crop.ch_names.index(ch)
    return ep_crop.get_data()[:, idx, :].mean(axis=1) * 1e6  # en µV

amp_compat   = mean_amplitude(compat_data,   tmin_stat, tmax_stat)
amp_incompat = mean_amplitude(incompat_data, tmin_stat, tmax_stat)

# T-test indépendant
t_stat, p_val = stats.ttest_ind(amp_compat, amp_incompat)

print(f"Compatible   : M = {amp_compat.mean():.2f} µV  (SD = {amp_compat.std():.2f})")
print(f"Incompatible : M = {amp_incompat.mean():.2f} µV  (SD = {amp_incompat.std():.2f})")
print(f"t = {t_stat:.3f}, p = {p_val:.4f}")
if p_val < 0.05:
    print("→ Différence significative ✓")
else:
    print("→ Différence non significative")

In [ ]:
import os
for f in ['raw_signal.png', 'erp.png', 'topomap.png']:
    print(f, "✓" if os.path.exists(f) else "✗ manquant")

In [ ]:
from google.colab import files
files.download('erp.png')
files.download('topomap.png')